In [3]:
import numpy as np
import pandas as pd
import time
import json
import os
import csv
from pulp import *

In [4]:
WLS_PARAMS = []
with open('gurobi.lic', 'r') as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        
        if '=' in line:
            key, val = line.split('=', 1)
            WLS_PARAMS.append((key.strip(),val.strip()))
GUROBI_EXE_PATH = r"C:\gurobi1301\win64\bin\gurobi_cl.exe" 

In [13]:
def save_experiment_json(exp_id, prob, x, P, alpha, beta, n_limit, penalties, matrix_file, prior_codes, penalty_mapping):
    solve_time = getattr(prob, 'solutionTime', 0)
    solver_name = getattr(prob, 'usedSolver', "GUROBI")
    status = LpStatus[prob.status]
    
    selected_duties = [j for j, var in x.items() if value(var) > 0.5]
    cancelled_pows = [i for i, var in P.items() if value(var) > 0.5]
    total_penalty = sum(penalties[i] for i in cancelled_pows)
    cancelled_codes_list = [prior_codes[i] for i in cancelled_pows]
    stats_simple = {
        code: cancelled_codes_list.count(code) 
        for code in ['CP', 'CO', 'NP', 'NO']
    }

    experiment_data = {
        "metadata": {
            "exp_id": exp_id,
            "matrix_used": matrix_file
        },
        "input_config": {
            "alpha": alpha,
            "beta": beta,
            "N_limit": n_limit,
            "penalty_vector": penalties,
            "penalty_mapping": penalty_mapping
        },
        "output_results": {
            "status": status,
            "solver": solver_name,
            "runtime_sec": round(solve_time, 4),
            "total_cost": value(prob.objective) if status == 'Optimal' else None,
            "drivers_used": len(selected_duties),
            "total_penalty": total_penalty,
            "cancelled_count": len(cancelled_pows),
            "selected_duty_indices": selected_duties,
            "cancelled_pow_indices": cancelled_pows,
            "cancleled_pow_code": cancelled_codes_list,
            "statistics_four_catagories": stats_simple
        }
    }

    json_output = json.dumps(experiment_data, indent=4)
    
    for key in ["penalty_vector", "selected_duty_indices", "cancelled_pow_indices", "cancleled_pow_code"]:
        json_output = re.sub(
            rf'("{key}":\s*)\[\s*(.*?)\s*\]',
            lambda m: m.group(1) + '[' + re.sub(r'\s+', ' ', m.group(2)) + ']',
            json_output,
            flags=re.DOTALL
        )

    if not os.path.exists("results"):
        os.makedirs("results")

    filename = f"results/{exp_id}_full_report.json"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(json_output)
        
    return experiment_data

In [6]:
def load_matrix(file_name):
    df = pd.read_csv(file_name, index_col=0)
    
    matrix_A = df.values
    
    m, n = matrix_A.shape
    
    return matrix_A, m, n

In [7]:
def read_csv_to_list(filename):
    with open(filename, mode='r', encoding='utf-8') as f:
        reader = csv.reader(f)
        data = next(reader) 
    return data

In [8]:
def build_and_solve_model(matrix_A, alpha, beta, n_limit, penalties):
    """
    Establish and solve the Set Covering model.
    """
    m, n = matrix_A.shape
    
    # --- 1. Initialize the model ---
    prob = LpProblem("Driver_Scheduling", LpMinimize)
    
    # --- 2. Define model variables ---
    x = LpVariable.dicts("Duty", range(n), cat='Binary')
    P = LpVariable.dicts("Penalty", range(m), cat='Binary')
    
    # --- 3. Objective function ---
    # min (alpha * sum x_j + beta * sum Pi * Ci)
    prob += (alpha * lpSum([x[j] for j in range(n)]) + 
             beta * lpSum([P[i] * penalties[i] for i in range(m)]))
    
    # --- 4. Constraints ---
    # A. Cover or Abandon: sum(x_ij) + Pi >= 1
    # Preprocessing: Find out which Duty j covers each PoW i to speed up modeling
    for i in range(m):
        covered_by_duties = [x[j] for j in range(n) if matrix_A[i][j] == 1]
        prob += lpSum(covered_by_duties) + P[i] >= 1
        
    # B. Max avaible drivers: sum(x_j) <= N
    prob += lpSum([x[j] for j in range(n)]) <= n_limit
    
    # --- 5. Solve ---
    start_solve = time()
    # prob.solve(PULP_CBC_CMD(msg=0))
    prob.solve(GUROBI_CMD(
        path=GUROBI_EXE_PATH, 
        msg=0,
        options=WLS_PARAMS
    ))
    # The solution time is recorded in the prob
    prob.solutionTime = time() - start_solve
    
    return prob, x, P

In [9]:
def run_experiment_pipeline(exp_id, alpha, beta, n_limit, matrix_file, penalties, prior_codes, penalty_mapping):

    # --- 1. Read data ---
    matrix_A, m, n = load_matrix(matrix_file)
    
    # --- 2. Establish and solve the model ---
    prob, x, P = build_and_solve_model(matrix_A, alpha, beta, n_limit, penalties)

    # --- 3. Call the saving function ---
    clean_name = os.path.basename(matrix_file).replace('.csv', '')
    report = save_experiment_json(
        exp_id, prob, x, P, 
        alpha, beta, n_limit, penalties, clean_name, prior_codes, penalty_mapping
    )
    
    # --- 4. Show brief results ---
    status = report['output_results']['status']
    total_cost = report['output_results']['total_cost']
    print(f"Experiment {exp_id} done！Status: {status}, Total cost: {total_cost}")

In [10]:
def run_stress_test_pipeline(alpha, beta, matrix_file, priority_file, penalty_mapping, test_plans, test_cat):

    # --- 1. Read data ---
    matrix_A, m, n = load_matrix(matrix_file)
    prior_codes = read_csv_to_list(priority_file)
    penalties = [penalty_mapping[code] for code in prior_codes]
    
    # --- 2. Find N_base ---
    print(f"--- 正在計算基準人力需求 (100% Coverage) ---")
    base_prob, base_x, base_P = build_and_solve_model(matrix_A, alpha, beta, n_limit=n, penalties=penalties)
    
    if LpStatus[base_prob.status] != 'Optimal':
        print("基準實驗未找到最優解，請檢查模型。")
        return

    # Calculate the actual number of drivers used
    n_base = sum(value(base_x[j]) for j in range(n) if value(base_x[j]) > 0.5)
    print(f"基準人力需求 N_base = {n_base}\n")

    # --- 3. Define all N values ​​to be tested ---    
    for pct in test_plans:
        current_n_limit = int(n_base * pct)
        
        # Dynamically generate experiment IDs
        clean_name = os.path.basename(matrix_file).replace('.csv', '')

        exp_id = f"{test_cat}_A{alpha}_B{beta}_N{current_n_limit}_{pct*100:.0f}pct_{clean_name}"
        
        print(f">>> 執行實驗: {exp_id} (人力: {current_n_limit}, 比例: {pct*100:.0f}%)")
        
        # Call single-experiment pipeline
        run_experiment_pipeline(
            exp_id=exp_id,
            alpha=alpha,
            beta=beta,
            n_limit=current_n_limit,
            matrix_file=matrix_file,
            penalties=penalties,
            prior_codes=prior_codes,
            penalty_mapping=penalty_mapping
        )
    
    print("\n✅ 所有壓力測試實驗已完成！")

In [11]:
EXP_ID = "EXP_test_pipeline" 
alpha = 1.0     # Driver weight
beta = 1.0      # Penalty weight
target_percentages = [1.0, 0.85, 0.75, 0.60]
penalty_mapping = {'CP': 3, 'CO': 3, 'NP': 1, 'NO': 1} # CN
# penalty_mapping = {'CP': 3, 'CO': 3, 'NP': 1, 'NO': 1} # PO
# penalty_mapping = {'CP': 9, 'CO': 3, 'NP': 3, 'NO': 1} # CNPO
matrix_file = '..\duty_generation\coverage_matrix_1_3_28_73.csv' # Input matrix
# matrix_file = 'test\mock_duty_matrix_100_500.csv'
priority_file = '..\duty_generation\output.csv' # Input priority list
# priority_file = 'test\output_column.csv'

In [12]:
matrix_A, m, n = load_matrix(matrix_file)
prior_codes = read_csv_to_list(priority_file)
print(m,n,len(prior_codes))

1013 20276 1013


In [14]:
run_stress_test_pipeline(alpha, beta, matrix_file, priority_file, penalty_mapping, target_percentages, 'CN')

--- 正在計算基準人力需求 (100% Coverage) ---
基準人力需求 N_base = 175.0

>>> 執行實驗: CN_A1.0_B1.0_N175_100pct_coverage_matrix_1_3_28_73 (人力: 175, 比例: 100%)
Experiment CN_A1.0_B1.0_N175_100pct_coverage_matrix_1_3_28_73 done！Status: Optimal, Total cost: 1043.0
>>> 執行實驗: CN_A1.0_B1.0_N148_85pct_coverage_matrix_1_3_28_73 (人力: 148, 比例: 85%)
Experiment CN_A1.0_B1.0_N148_85pct_coverage_matrix_1_3_28_73 done！Status: Optimal, Total cost: 1067.0
>>> 執行實驗: CN_A1.0_B1.0_N131_75pct_coverage_matrix_1_3_28_73 (人力: 131, 比例: 75%)
Experiment CN_A1.0_B1.0_N131_75pct_coverage_matrix_1_3_28_73 done！Status: Optimal, Total cost: 1101.0
>>> 執行實驗: CN_A1.0_B1.0_N105_60pct_coverage_matrix_1_3_28_73 (人力: 105, 比例: 60%)
Experiment CN_A1.0_B1.0_N105_60pct_coverage_matrix_1_3_28_73 done！Status: Optimal, Total cost: 1162.0

✅ 所有壓力測試實驗已完成！
